In [25]:
import json
from pathlib import Path
import pandas as pd
import re
import extract as ext
from shutil import move

In [26]:
ROOT = Path("~/code/selector_examenes").expanduser()

In [27]:
form_classified = pd.read_csv("form_classified.csv", encoding="utf-8")
form_emestrada = pd.read_csv("form_emestrada.csv", encoding="utf-8")

In [28]:
form_classified.loc[form_classified.year == 2018, "statement"]

98     1.- Formule o nombre los siguientes compuestos...
99     1.- Formule o nombre los siguientes compuestos...
100    1.- Formule o nombre los siguientes compuestos...
101    1.- Formule o nombre los siguientes compuestos...
102    1.- Formule o nombre los siguientes compuestos...
103    1.- Formule o nombre los siguientes compuestos...
104    1.- Formule o nombre los siguientes compuestos...
105    1.- Formule o nombre los siguientes compuestos...
106    1.- Formule o nombre los siguientes compuestos...
107    1.- Formule o nombre los siguientes compuestos...
108    1.- Formule o nombre los siguientes compuestos...
109    1.- Formule o nombre los siguientes compuestos...
Name: statement, dtype: str

In [29]:
form_emestrada.head()

,year,exam_name,statement
0,2010,química - 2010 - junio,a) Nitrito de plata b) Hidróxido de magnesio c...
1,2010,química - 2010 - junio,a) Peróxido de rubidio b) Hidrogenocarbonato d...
2,2010,química - 2010 - reserva 1,a) Permanganato de bario b) Peróxido de potasi...
3,2010,química - 2010 - reserva 1,a) Hidrogenocarbonato de sodio b) Hidróxido de...
4,2010,química - 2010 - reserva 2,a) Pentafluoruro de antimonio b) Óxido de plom...


In [30]:
form_classified.head()

,year,exam_name,exercise_number,topic,exercise_type,statement,exam_option,exam_file_path
0,2010,Quimica - quimica - quimica - junio - quimica ...,1,Formulación,Formulación,1.- Formule o nombre los compuestos siguientes...,Opción A,examenes_quimica/2010_quimica/2010_quimica_qui...
1,2010,Quimica - quimica - quimica - junio - quimica ...,1,Formulación,Formulación,1.- Formule o nombre los compuestos siguientes...,Opción A,examenes_quimica/2010_quimica/2010_quimica_qui...
2,2010,Quimica - quimica - quimica - junio - quimica ...,1,Formulación,Formulación,1.- Formule o nombre los compuestos siguientes...,Opción A,examenes_quimica/2010_quimica/2010_quimica_qui...
3,2010,Quimica - quimica - quimica - junio - quimica ...,1,Formulación,Formulación,1.- Formule o nombre los compuestos siguientes...,Opción A,examenes_quimica/2010_quimica/2010_quimica_qui...
4,2010,Quimica - quimica - quimica - junio - quimica ...,1,Formulación,Formulación,1.- Formule o nombre los compuestos siguientes...,Opción A,examenes_quimica/2010_quimica/2010_quimica_qui...


In [31]:
form_classified.year.aggregate(["min", "max"])

min    2010
max    2026
Name: year, dtype: int64

In [32]:
form_emestrada.year.aggregate(["min", "max"])

min    2010
max    2025
Name: year, dtype: int64

In [33]:
for row in form_emestrada.statement[:3]:
    print(row)

a) Nitrito de plata b) Hidróxido de magnesio c) 1,1 Dicloroetano
a) Peróxido de rubidio b) Hidrogenocarbonato de calcio c) Butanona
a) Permanganato de bario b) Peróxido de potasio c) Triclorometano


In [34]:
for row in form_classified.statement[:3]:
    print(row)

1.- Formule o nombre los compuestos siguientes: a) Nitrito de plata b) Hidróxido de magnesio
c) 1,1-Dicloroetano d) MoO₃ e) Ca₃(PO₄)₂ f) CH₂OHCH₂OH
1.- Formule o nombre los compuestos siguientes: a) Peróxido de rubidio b) Hidrogenocarbonato de calcio
c) Butanona d) BeH₂ e) HClO₄ f) CH₃CONH₂
1.- Formule o nombre los compuestos siguientes: a) Permanganato de bario b) Peróxido de potasio
c) Triclorometano d) HgO e) ZnS f) CH₂=CHCH₃


Limpiar y adaptar los enunciados de formulación de `form_classified`

In [35]:
def extraer_nombrar_compuestos_25(statement):
    if not pd.isna(statement):
        start_position = statement.find("c)")
        statement = statement[start_position:]
        statement = statement.replace(";", "").replace("\n", " ").strip()
    
    return statement

In [40]:
form_classified.loc[form_classified.year == 2025, "statement"] = form_classified.loc[form_classified.year == 2025, "statement"].apply(extraer_nombrar_compuestos_25)

In [36]:
def extraer_nombrar_compuestos(statement):
    if not pd.isna(statement):
        start_position = statement.find("a)")
        if " d) " in statement:
            end_position = statement.find(" d) ")
        else:
            end_position = -1
        statement = statement[start_position:end_position]
        # unir valencia con Stock
        statement = re.sub(r"\s+(\([IVX]+\))", r"\1", statement)
        statement = statement.replace(";", "").replace("\n", " ").strip()
        #statement = re.sub(r"[^0-9a-záéíóú]", "", statement.lower())
    
    return statement

In [41]:
form_classified.loc[form_classified.year != 2025, "statement"] = form_classified.loc[form_classified.year != 2025, "statement"].apply(extraer_nombrar_compuestos)

In [43]:
for row in form_classified.statement[:3]:
    print(extraer_nombrar_compuestos(row))

a) Nitrito de plata b) Hidróxido de magnesio c) 1,1-Dicloroetan
a) Peróxido de rubidio b) Hidrogenocarbonato de calcio c) Butanon
a) Permanganato de bario b) Peróxido de potasio c) Triclorometan


In [44]:
# x: classified, y: emestrada
form_merged = pd.merge(form_classified, form_emestrada, on=["statement", "year"])

In [45]:
form_merged.shape

(121, 9)

In [47]:
form_merged.columns

Index(['year', 'exam_name_x', 'exercise_number', 'topic', 'exercise_type',
       'statement', 'exam_option', 'exam_file_path', 'exam_name_y'],
      dtype='str')

In [48]:
form_merged.year.aggregate(["min", "max"])

min    2010
max    2022
Name: year, dtype: int64

In [49]:
form_merged.head(1).T

,0
year,2010
exam_name_x,Quimica - quimica - quimica - junio - quimica ...
exercise_number,1
topic,Formulación
exercise_type,Formulación
statement,a) Peróxido de rubidio b) Hidrogenocarbonato d...
exam_option,Opción A
exam_file_path,examenes_quimica/2010_quimica/2010_quimica_qui...
exam_name_y,química - 2010 - junio


In [50]:
cols_to_keep = ["year", "exam_file_path", "exam_name_y"]
form_merged = form_merged.drop([col for col in form_merged.columns if col not in cols_to_keep], axis = 1)

In [51]:
form_merged = form_merged.rename(columns = {"exam_name_y" : "filename_emestrada",
                     "exam_file_path" : "source_file_path",})

In [52]:
form_merged.head()

,year,source_file_path,filename_emestrada
0,2010,examenes_quimica/2010_quimica/2010_quimica_qui...,química - 2010 - junio
1,2010,examenes_quimica/2010_quimica/2010_quimica_qui...,química - 2010 - reserva 1
2,2010,examenes_quimica/2010_quimica/2010_quimica_qui...,química - 2010 - reserva 1
3,2010,examenes_quimica/2010_quimica/2010_quimica_qui...,química - 2010 - reserva 2
4,2010,examenes_quimica/2010_quimica/2010_quimica_qui...,química - 2010 - septiembre


In [53]:
form_merged = form_merged.drop_duplicates()

## Renombrar archivos

In [54]:
for idx, row in form_merged.iterrows():
    old = ROOT / row.source_file_path
    new = old.parent / f"{row.filename_emestrada}.pdf"

    #print(f"{old} --> {new}")
    try:
        move(old, new)
    except Exception as e:
        print(f"Archivo: {old.name}. Error: {e}")